# From Cube to SLayer

**TL;DR:** SLayer can import [Cube](https://cube.dev) data models — cubes and views, in **YAML or JavaScript** — and turn them into queryable SLayer datasets. This notebook imports a small jaffle-shop-flavored Cube project end-to-end with `slayer import-cube`, then queries the result and checks every answer against gold SQL.

Cube import works **offline**: with no reachable datasource at import time, column types come from Cube's own dimension / measure declarations (the report notes they were not checked against the database). The DuckDB below exists only so we can *query* the imported datasets.

Five steps (everything generated lands in a gitignored `.cache/` next to this notebook):

1. **Build the data** — a tiny retail DuckDB (orders, customers, products) with deterministic rows.
2. **The Cube configs** — two YAML cubes, a view, and a JS cube using `FILTER_PARAMS`.
3. **Reference answers** — gold SQL, run up front.
4. **Import** — run `slayer import-cube`.
5. **Inspect & query** — inspect the generated datasets and query them — including the `FILTER_PARAMS` pushdowns.

See also: [Importing Cube definitions](../../cube/cube_import.md) · [Variable Substitution](../14_variable_substitution/variable_substitution_nb.ipynb) — the `{var}` / `{? ?}` mechanics imported datasets rely on.

## Step 1 — Build the demo database

Create the retail DuckDB the cubes bind to. Cube import itself needs no database — but we query the imported datasets afterwards, so the data has to exist.

In [ ]:
import json
import os
import sys

import pandas as pd

# The setup helper lives next to this notebook.
sys.path.insert(0, os.getcwd())

from setup_cube import (
    build_shop_duckdb,
    compute_gold,
    import_cube_cli,
    save_datasource,
    CUBE_PROJECT,
    DB_PATH,
    DATASETS_DIR,
    REPORT_PATH,
    DATASOURCE_NAME,
)
from slayer.async_utils import run_sync
from slayer.engine.query_engine import SlayerQueryEngine
from slayer.inspect.dataset_render import render_dataset_skeleton
from slayer.storage.yaml_storage import YAMLStorage

db_path = build_shop_duckdb(DB_PATH)
print(f"Built retail DuckDB '{db_path.name}' with orders, customers, products.")

## Step 2 — The Cube configs

The project has four files. `orders` and `customers` are ordinary table-anchored cubes (with a join between them); `orders_overview` is a **view** (it owns no table); and `order_facts` is a **JavaScript** cube whose raw SQL uses `FILTER_PARAMS` — Cube's mechanism for caller-supplied filters. Watch the two `FILTER_PARAMS` lines: `category` carries `meta.required` (a *required* pushdown), `region` does not (an *optional* one).

In [ ]:
for rel in [
    "model/cubes/orders.yml",
    "model/cubes/customers.yml",
    "model/views/orders_overview.yml",
    "model/cubes/order_facts.js",
]:
    print(f"# ===== {rel} =====")
    print((CUBE_PROJECT / rel).read_text().rstrip())
    print()

## Step 3 — Reference answers (gold SQL)

Run the gold SQL **first**, up front, and stash the expected numbers. SLayer opens the DuckDB file with a read-write engine that a second raw connection can't share, so every gold query runs *before* any SLayer query touches the file.

In [ ]:
GOLD = compute_gold(DB_PATH)

print("total amount (all)       :", GOLD["total"])
print("order_facts count (all)  :", GOLD["of_all"])
print("count region=North       :", GOLD["of_north"])
print("count region=South       :", GOLD["of_south"])
print("count category=Beverages :", GOLD["of_beverages"])
pd.DataFrame(GOLD["by_region"])

## Step 4 — Import with `slayer import-cube`

This is the one-liner an operator runs:

```bash
slayer import-cube cube_project --datasource shop_cube --storage .cache/slayer_datasets
```

It reads every `.yml` / `.yaml` / `.js` under the path, converts each cube and view to a SLayer dataset, and writes a JSON report. Below we run exactly that command (through the current interpreter) and print its summary. The two `FILTER_PARAMS` members show up as an *optional* (`region`) and a *required* (`category`) variable.

In [ ]:
proc = import_cube_cli(datasets_dir=DATASETS_DIR, report_path=REPORT_PATH)
print(proc.stdout.replace(str(REPORT_PATH.resolve()), REPORT_PATH.name).rstrip())

report = json.loads(REPORT_PATH.read_text())
errors = [i for i in report["issues"] if i["severity"] == "error"]
assert not errors, errors

`import-cube` files the datasets under the datasource **name** but doesn't create the datasource itself. Querying needs it registered — and because the import wipes the storage directory first, we register it now, *after* importing.

In [ ]:
save_datasource(DATASETS_DIR, DB_PATH)
print("Registered datasource:", DATASOURCE_NAME)

## Step 5 — Inspect the generated datasets

`order_facts` became an **sql-mode** dataset whose `FILTER_PARAMS` turned into `{var}` placeholders. The dataset skeleton's `Variables:` line classifies them — `category (required)` vs a bare optional `region` — and `meta.cube_variables` records the per-variable contract the engine reads (`list_valued`, `required`, Cube `kind`).

In [ ]:
storage = YAMLStorage(base_dir=str(DATASETS_DIR))
order_facts = run_sync(storage.get_dataset("order_facts", data_source=DATASOURCE_NAME))

print("Dataset skeleton for 'order_facts':\n")
print(render_dataset_skeleton(dataset=order_facts))

print("\nmeta.cube_variables:")
for name, spec in order_facts.meta["cube_variables"].items():
    print(f"  {name}: required={spec['required']}, list_valued={spec['list_valued']}, kind={spec['kind']!r}")

## Query the view and the relationship

Point the engine at the imported datasets. `orders_overview` (the view) and the raw `orders` cube both reach `region` on `customers` — through the relationship the importer inferred from the Cube config. No SQL join is written by hand.

In [ ]:
engine = SlayerQueryEngine(storage=storage)
gold_by_region = {g["region"]: g["amount"] for g in GOLD["by_region"]}

view_rows = engine.execute_sync(query={
    "population": "orders_overview",
    "metrics": ["orders_overview.total_amount"],
    "dimensions": ["orders_overview.region"],
    "order": [{"field": "orders_overview.region"}],
}).data
display(pd.DataFrame(view_rows))

got = {r["orders_overview.region"]: r["orders_overview.total_amount"] for r in view_rows}
assert got == gold_by_region, f"{got} != {gold_by_region}"
print("OK — view total by region matches gold:", gold_by_region)

In [ ]:
relationship_rows = engine.execute_sync(query={
    "population": "orders",
    "metrics": ["orders.total_amount"],
    "dimensions": ["customers.region"],
    "order": [{"field": "customers.region"}],
}).data

got = {r["customers.region"]: r["orders.total_amount"] for r in relationship_rows}
assert got == gold_by_region
print("OK — the raw `orders` cube reaches `customers.region` via the imported relationship:", got)

## The `FILTER_PARAMS` pushdowns

`order_facts` takes two caller-supplied filters. `category` is **required** (a bare `p.category IN ({category})`); `region` is **optional** — wrapped in a block `{? c.region IN ({region}) ?}` that collapses to `(1=1)` when omitted. Both are set-membership (`IN`) filters, so the importer marked them `list_valued`: pass a list, or a bare scalar the engine wraps into a one-element list.

In [ ]:
BOTH = ["Beverages", "Bakery"]


def count(**variables):
    r = engine.execute_sync(query={
        "population": "order_facts",
        "metrics": [{"metric": "order_facts.count"}],
        "variables": variables,
    })
    return r.data[0]["order_facts.count"], r.sql


def where_lines(sql, *needles):
    return "\n".join(ln.rstrip() for ln in sql.splitlines() if any(n in ln for n in needles))


def show(label, got, gold):
    print(f"{label}: {got}   (gold {gold})")
    assert got == gold, f"{label}: {got} != {gold}"

### Optional omitted → the block collapses to `(1=1)`

Supply the required `category` (both values, so nothing is excluded) and omit `region`. The optional block disappears and every order counts.

In [ ]:
n, sql = count(category=BOTH)  # region omitted
show("count (region omitted)", n, GOLD["of_all"])
print("\nThe optional `region` block collapsed to (1 = 1):")
print(where_lines(sql, "1 = 1", "IN ("))

### Optional supplied (a list) → a real `IN (...)`

Now pass `region=["North"]`. The block renders and filters.

In [ ]:
n_north, sql = count(category=BOTH, region=["North"])
show("count region=['North']", n_north, GOLD["of_north"])
print("\nRendered filter:")
print(where_lines(sql, "region", "IN ("))

### A scalar for a list-valued variable → coerced to a one-element list

The importer wrote the `IN (...)` parentheses, so a caller has nowhere to put per-element quotes. SLayer therefore wraps a bare scalar into a one-element list — `region="North"` means exactly `region=["North"]`.

In [ ]:
n_scalar, _ = count(category=BOTH, region="North")   # bare string, not a list
print("region='North' (scalar):", n_scalar, "  region=['North'] (list):", n_north)
assert n_scalar == n_north

n_bev, _ = count(category="Beverages")   # scalar for the required var; region omitted
show("category='Beverages' (scalar)", n_bev, GOLD["of_beverages"])

### The required pushdown is not optional

Omit `category` (supplying only `region`) and the query raises, naming the missing variable — a required pushdown fails loudly rather than matching nothing.

In [ ]:
try:
    count(region=["North"])   # category (required) omitted
except ValueError as e:
    print("Omitting the required `category` raises:")
    print(" ", e)

## Recap

Starting from a Cube project we did not write, `slayer import-cube`:

- turned two YAML cubes and a view into queryable SLayer datasets, with a relationship inferred from the Cube config,
- converted a JS cube's `FILTER_PARAMS` into SLayer `{var}` substitution — a **required** pushdown (`category`) and an **optional** one (`region`) that collapses to `(1=1)` when omitted,
- marked both set-membership pushdowns `list_valued`, so a bare scalar is coerced to a one-element list.

Every answer matched gold SQL.

### Further reading

- [Importing Cube definitions](../../cube/cube_import.md) — the full conversion reference (what maps, what fails cleanly, the report).
- [Variable Substitution](../14_variable_substitution/variable_substitution_nb.ipynb) — the `{var}` / `{? ?}` mechanics, across all of SLayer's raw-SQL surfaces.